# Topic-wise generation pipeline (uniform static greenlists)

First layer of the dual scheme, now driven by the **static** uniform lists exported
by `05_greenlist_construction.ipynb`: one CSV per topic under
`data/greenlist/<topic>.csv` with columns

    token_id, token, similarity, type ∈ {content, connector, residual}

Connectors *and* below-threshold residuals are round-robined across topics, so every
list covers the whole vocabulary (γ = |G_t| / vocab ≈ 0.13–0.25 instead of the old
0.002–0.14).

Flow:
1. config + model load (OPT-2.7b — same embeddings that built the lists)
2. load uniform greenlists from `data/greenlist/`
3. topic inference: mean-pooled prompt embedding vs topic vectors
   (same geometry as notebooks 04/05)
4. `TopicBoostProcessor` boosts the assigned topic's greenlist at every step
5. plain vs watermarked output, side by side

In [22]:
import glob
import os

import pandas as pd
import torch
from transformers import AutoTokenizer, LogitsProcessor, OPTForCausalLM

# ---------------------------------------------------------------- config ----
# for cand in ("../drive/MyDrive/minor_project/green_list_topics_uniform", ""):
GREENLIST_DIR = "greenlist"

MODEL_NAME     = "facebook/opt-2.7b"
DELTA          = 2.0        # topic-layer boost (recalibrate for uniform lists)
MAX_NEW_TOKENS = 200        # dual-layer operating length (PLAN.md)
TEMPERATURE    = 1.0
TOP_P          = 0.9
SEED           = 0
SPLIT          = "all"      # "all" = paper-style full coverage, "content" = strict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

TOPICS = sorted(os.path.splitext(os.path.basename(p))[0]
                for p in glob.glob(os.path.join(GREENLIST_DIR, "*.csv")))
print("device:", DEVICE)
print("greenlist dir:", GREENLIST_DIR)
print("topics:", TOPICS)

device: cuda
greenlist dir: greenlist
topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']


In [23]:
# ---------------------------------------------------------------- model -----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = OPTForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()

# NOTE: OPT quirk -- len(tokenizer) = 50265 usable ids, but the embedding
# matrix / config.vocab_size = 50272 rows (7 reserved slots, never emitted).
# Greenlists were built over the tokenizer id-space (opt27_vocab.csv),
# so len(tokenizer) is the correct denominator for gamma.
emb_rows = model.get_input_embeddings().weight.shape[0]
assert emb_rows >= len(tokenizer)
VOCAB_SIZE = len(tokenizer)
print(f"usable ids: {len(tokenizer)} | embedding rows: {emb_rows}")
print("vocab size (gamma denominator):", VOCAB_SIZE)

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

usable ids: 50265 | embedding rows: 50272
vocab size (gamma denominator): 50265


In [ ]:
# ------------------------------------------------ static uniform greenlists -
def load_uniform_greenlists(greenlist_dir):
    """One CSV per topic -> {topic: {"all": [ids], "content": [ids]}}."""
    out = {}
    for path in sorted(glob.glob(os.path.join(greenlist_dir, "*.csv"))):
        topic = os.path.splitext(os.path.basename(path))[0]
        df = pd.read_csv(path)
        assert df["token_id"].is_unique, f"duplicate ids in {path}"
        out[topic] = {
            "all": df["token_id"].astype(int).tolist(),
            "content": df.loc[df["type"] == "content",
                              "token_id"].astype(int).tolist(),
        }
    return out


greenlists = load_uniform_greenlists(GREENLIST_DIR)
print({t: len(v["all"]) for t, v in greenlists.items()})

In [24]:

@torch.no_grad()
def build_normed_embeddings():
    emb = model.get_input_embeddings().weight.detach().float()
    return emb / emb.norm(dim=1, keepdim=True)


normed_embeddings = build_normed_embeddings()

topic_token_ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0]
                   for t in TOPICS]
for t, tid in zip(TOPICS, topic_token_ids):
    print(f"{t!r:14} -> {tokenizer.decode([tid])!r} (id {tid})")

topic_matrix = normed_embeddings[topic_token_ids]   # (K, d), unit-norm rows

'entertainment' -> ' entertainment' (id 4000)
'finance'      -> ' finance' (id 2879)
'history'      -> ' history' (id 750)
'medicine'     -> ' medicine' (id 6150)
'politics'     -> ' politics' (id 2302)
'science'      -> ' science' (id 2866)
'sports'       -> ' sports' (id 1612)
'technology'   -> ' technology' (id 806)


In [25]:

@torch.no_grad()
def rank_topics(prompt):
    """Cosine similarity of mean-pooled prompt embedding against each topic."""
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    prompt_vec = normed_embeddings[ids].mean(dim=0)
    prompt_vec = prompt_vec / prompt_vec.norm()
    scores = topic_matrix @ prompt_vec                 # (K,)
    return sorted(zip(TOPICS, scores.tolist()), key=lambda x: -x[1])


def extract_topic(prompt, show=False):
    ranked = rank_topics(prompt)
    if show:
        for name, s in ranked:
            bar = "#" * max(0, int((s + 1) * 20))      # cosine [-1, 1] -> bar
            print(f"  {name:14} {s:+.4f} {bar}")
    return ranked[0][0], ranked[0][1]


# extract_topic("The government passed a new laws and laws poltical law and legistature before the election", show=True)

## First layer — public, topic-based watermark

| | second layer (private KGW) | first layer (public topic) |
|---|---|---|
| greenlist | derived from key + prev tokens | **static**, fixed per topic (CSV) |
| selection | hash-based | topic extracted from prompt |
| boost | `scores[b, preferred] += delta_private` | `scores[..., topic_green_ids] += delta` |

Flow: `extract_topic(prompt)` -> assign -> `TopicBoostProcessor` boosts that
topic's greenlist logits at every generation step. Because the lists are uniform,
every emitted token has a key hit — detection stays well-defined everywhere.

In [26]:
# ---------------------------------------------------------------- watermark -
class TopicBoostProcessor(LogitsProcessor):


    def __init__(self, green_token_ids, delta=DELTA):
        self.ids = torch.tensor(green_token_ids, dtype=torch.long)
        self.delta = delta

    def __call__(self, input_ids, scores):
        scores[..., self.ids.to(scores.device)] += self.delta
        return scores

In [27]:
# ------------------------------------------------------------- generation ---
GEN_KWARGS = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                  temperature=TEMPERATURE, top_p=TOP_P)


@torch.no_grad()
def generate_pair(prompt, topic=None, split=SPLIT):
    """Plain vs topic-watermarked continuation of `prompt`. Returns a dict."""
    if topic is None:
        topic, _ = extract_topic(prompt)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    plain = model.generate(**inputs, **GEN_KWARGS)
    boosted = model.generate(
        **inputs, **GEN_KWARGS,
        logits_processor=[TopicBoostProcessor(greenlists[topic][split])],
    )
    return {
        "topic": topic,
        "plain_output": tokenizer.decode(plain[0], skip_special_tokens=True),
        "watermarked_output": tokenizer.decode(boosted[0], skip_special_tokens=True),
    }


res = generate_pair("The government passed a new law before the election")
print("topic :", res["topic"])
print("PLAIN :", res["plain_output"])
print("WM    :", res["watermarked_output"])

topic : history
PLAIN : The government passed a new law before the election. You may have to show ID to vote.
I just registered to vote.
WM    : The government passed a new law before the election so that they could use it as a justification for the censorship and surveillance after they took over. It was the worst of both worlds. It's like they're scared they'll look bad in the eyes of the public so they want to have enough tools in the bag that they know how to use them at any point in the future.
"We could've done this without any new laws or regulations"


In [28]:
# -------------------------------------------------------------- batch run ---
PROMPTS_PER_TOPIC = {
    "technology":    "The new AI chip runs twice as fast while using less power",
    "medicine":      "Doctors tested the new vaccine across three hospitals",
    "sports":        "The cricket team won the final match on the last ball",
    "politics":      "The government passed a new law before the election",
    "science":       "Scientists discovered a new particle at the collider",
    "entertainment": "The movie sequel broke box office records opening weekend",
    "finance":       "Central bank rates pushed markets down this quarter",
    "history":       "Ancient Rome fell after centuries of decline",
}

rows = []
for i, prompt in enumerate(PROMPTS_PER_TOPIC.values()):
    topic, score = extract_topic(prompt)
    result = generate_pair(prompt, topic=topic)
    rows.append({
        "id": i,
        "prompt": prompt,
        "topic": topic,
        "topic_score": round(score, 4),
        "delta": DELTA,
        "seed": SEED,
        "split": SPLIT,
        "max_new_tokens": MAX_NEW_TOKENS,
        **result,
    })

results = pd.DataFrame(rows)
results

,id,prompt,topic,topic_score,delta,seed,split,max_new_tokens,plain_output,watermarked_output
0,0,The new AI chip runs twice as fast while using...,technology,0.2613,2.0,0,all,200,The new AI chip runs twice as fast while using...,The new AI chip runs twice as fast while using...
1,1,Doctors tested the new vaccine across three ho...,medicine,0.2475,2.0,0,all,200,Doctors tested the new vaccine across three ho...,Doctors tested the new vaccine across three ho...
2,2,The cricket team won the final match on the la...,sports,0.2613,2.0,0,all,200,The cricket team won the final match on the la...,The cricket team won the final match on the la...
3,3,The government passed a new law before the ele...,history,0.2499,2.0,0,all,200,The government passed a new law before the ele...,The government passed a new law before the ele...
4,4,Scientists discovered a new particle at the co...,science,0.2634,2.0,0,all,200,Scientists discovered a new particle at the co...,Scientists discovered a new particle at the co...
5,5,The movie sequel broke box office records open...,history,0.2193,2.0,0,all,200,The movie sequel broke box office records open...,The movie sequel broke box office records open...
6,6,Central bank rates pushed markets down this qu...,history,0.1788,2.0,0,all,200,Central bank rates pushed markets down this qu...,Central bank rates pushed markets down this qu...
7,7,Ancient Rome fell after centuries of decline,history,0.2325,2.0,0,all,200,Ancient Rome fell after centuries of decline. ...,Ancient Rome fell after centuries of decline. ...


In [29]:
print(results.iloc[0]['prompt'], '\n')
print(results.iloc[0]['watermarked_output'], '\n')
print(results.iloc[0]['plain_output'])



The new AI chip runs twice as fast while using less power 

The new AI chip runs twice as fast while using less power than the last
Intel has officially unveiled a new processor architecture they call Comet Lake. Compared to the Kaby Lake architecture, Comet Lake delivers double the clock speed, doubles the cache and uses lower power. The new architecture will be available in processors ranging from the i3-9650C to the i7-10710.
Intel claims the new processors will be the fastest in the market. They say they also deliver four times the performance compared to their predecessor. They will also be faster than the AMD Ryzen CPUs.
Intel's new architecture uses the same technology that they used to develop the 28-nanometer Ice Lake chips. In fact the architecture is so similar to Ice Lake that the architecture team called it a "true leap forward."
The new architecture uses the same processors on the same chip. This is what enables the CPUs to be faster while using less power. They have also

In [ ]:
# make sure %%writefile below has a target directory
os.makedirs("../src/watermark", exist_ok=True)
print("export target:", os.path.abspath("../src/watermark/first_layer.py"))

## Export — `src/watermark/first_layer.py`

Packages the working cells above into the first-layer module, structured like
`second_layer.py` (processor class + high-level class with `generate`):

- `TopicBoostProcessor(green_ids, delta)` — the watermark itself
- `TopicWiseWatermarking(model, tokenizer, greenlist_dir=...)` —
  `.extract_topic(prompt)` · `.generate(prompt[, topic])` · `.generate_batch(prompts)`
- legacy `load_topic_greenlists` / `build_topic_matrix` kept so
  `src/watermark/dual_layer.py` imports keep working

Path assumes the notebook runs from `notebooks/`; drop the leading `../`
if your cwd is the repo root.

In [31]:
%%writefile first_layer.py
'''First layer: public, topic-based watermarking over STATIC uniform greenlists.

Each topic has a fixed greenlist CSV under data/greenlist/<topic>.csv
(columns: token_id, token, similarity, type in {content, connector, residual});
connectors and below-threshold residuals were round-robined across topics, so
every list covers the whole vocabulary.

The prompt topic is inferred by cosine similarity in OPT's own embedding space
(the same geometry that built the lists); the assigned topic's greenlist is then
boosted during sampling. Structure mirrors second_layer.py.
'''
import glob
import os

import pandas as pd
import torch
from transformers import LogitsProcessor


class TopicBoostProcessor(LogitsProcessor):
    """Adds `delta` to one topic's greenlist logits at every generation step."""

    def __init__(self, green_token_ids, delta):
        self.ids = torch.tensor(green_token_ids, dtype=torch.long)
        self.delta = delta

    def __call__(self, input_ids, scores):
        scores[..., self.ids.to(scores.device)] += self.delta
        return scores


def load_uniform_greenlists(greenlist_dir):
    """One CSV per topic -> {topic: {"all": [ids], "content": [ids]}}."""
    out = {}
    for path in sorted(glob.glob(os.path.join(greenlist_dir, "*.csv"))):
        topic = os.path.splitext(os.path.basename(path))[0]
        df = pd.read_csv(path)
        assert df["token_id"].is_unique, f"duplicate ids in {path}"
        out[topic] = {
            "all": df["token_id"].astype(int).tolist(),
            "content": df.loc[df["type"] == "content",
                              "token_id"].astype(int).tolist(),
        }
    return out


def load_topic_greenlists(csv_path):
    """Legacy single-CSV loader (kept for dual_layer.py compatibility)."""
    df = pd.read_csv(csv_path)
    return {t: g["token_id"].astype(int).tolist()
            for t, g in df.groupby("topic")}


def build_topic_matrix(model, tokenizer, topics):
    """(K, d) unit-norm rows: OPT input embedding of each topic's first subtoken."""
    emb = model.get_input_embeddings().weight.detach().float()
    emb = emb / emb.norm(dim=1, keepdim=True)
    ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in topics]
    return emb[ids]


class TopicWiseWatermarking:
    """Public topic watermark: route the prompt, boost its static greenlist.

    extract_topic(prompt)   -> (best_topic, ranked [(topic, cos_sim), ...])
    generate(prompt)        -> {"topic", "topic_score",
                                "plain_output", "watermarked_output"}
    generate_batch(prompts) -> pd.DataFrame of generate() dicts (+ id column)
    """

    def __init__(self, model, tokenizer, greenlist_dir="data/greenlist",
                 delta=4.0, split="all", max_new_tokens=200,
                 temperature=1.0, top_p=0.9, seed=0):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.delta = delta
        self.split = split
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p

        if seed is not None:
            torch.manual_seed(seed)

        self.greenlists = load_uniform_greenlists(greenlist_dir)
        self.topics = sorted(self.greenlists)

        emb = self.model.get_input_embeddings().weight.detach().float()
        self.normed_embeddings = emb / emb.norm(dim=1, keepdim=True)
        self.topic_matrix = self.normed_embeddings[
            [self.tokenizer.encode(" " + t, add_special_tokens=False)[0]
             for t in self.topics]
        ]

    @torch.no_grad()
    def extract_topic(self, prompt):
        ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        prompt_vec = self.normed_embeddings[ids].mean(dim=0)
        prompt_vec = prompt_vec / prompt_vec.norm()
        scores = self.topic_matrix @ prompt_vec
        ranked = sorted(zip(self.topics, scores.tolist()), key=lambda x: -x[1])
        return ranked[0][0], ranked

    def _generate(self, inputs, processor=None):
        kwargs = dict(max_new_tokens=self.max_new_tokens, do_sample=True,
                      temperature=self.temperature, top_p=self.top_p)
        if processor is not None:
            kwargs["logits_processor"] = [processor]
        with torch.no_grad():
            out = self.model.generate(**inputs, **kwargs)
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def generate(self, prompt, topic=None):
        """Plain vs watermarked continuation of `prompt`.

        topic=None infers it via extract_topic; pass an explicit topic to
        force routing (e.g. for controlled experiments).
        """
        if topic is None:
            topic, ranked = self.extract_topic(prompt)
            topic_score = round(ranked[0][1], 4)
        else:
            topic_score = None

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        processor = TopicBoostProcessor(self.greenlists[topic][self.split],
                                        self.delta)
        return {
            "topic": topic,
            "topic_score": topic_score,
            "plain_output": self._generate(inputs),
            "watermarked_output": self._generate(inputs, processor),
        }

    def generate_batch(self, prompts):
        rows = [{"id": i, "prompt": p, **self.generate(p)}
                for i, p in enumerate(prompts)]
        return pd.DataFrame(rows)


Writing first_layer.py


In [32]:
import importlib
import sys
sys.path.insert(0, "../src/watermark")

import first_layer
importlib.reload(first_layer)

twm = first_layer.TopicWiseWatermarking(
    model, tokenizer,
    greenlist_dir=GREENLIST_DIR,
    delta=DELTA, split=SPLIT,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE, top_p=TOP_P, seed=SEED,
)

check = twm.generate_batch(list(PROMPTS_PER_TOPIC.values()))

same_routing = check["topic"].tolist() == results["topic"].tolist()
print(f"module reproduces notebook topic routing: {same_routing}")
assert same_routing, "exported module disagrees with the notebook"

check.head()

module reproduces notebook topic routing: True


,id,prompt,topic,topic_score,plain_output,watermarked_output
0,0,The new AI chip runs twice as fast while using...,technology,0.2613,The new AI chip runs twice as fast while using...,The new AI chip runs twice as fast while using...
1,1,Doctors tested the new vaccine across three ho...,medicine,0.2475,Doctors tested the new vaccine across three ho...,Doctors tested the new vaccine across three ho...
2,2,The cricket team won the final match on the la...,sports,0.2613,The cricket team won the final match on the la...,The cricket team won the final match on the la...
3,3,The government passed a new law before the ele...,history,0.2499,The government passed a new law before the ele...,The government passed a new law before the ele...
4,4,Scientists discovered a new particle at the co...,science,0.2634,Scientists discovered a new particle at the co...,Scientists discovered a new particle at the co...
